# Kiểm thử End-to-End: Sinh Text bằng LLaVA và Dự đoán bằng MulCo
Mô phỏng pipeline hệ thống thực tế: 
1. Nhận ảnh gốc đầu vào (không có text đi kèm).
2. Gọi mô hình LLaVA để sinh văn bản mô tả đặc điểm, triệu chứng trên lá.
3. Đưa ảnh và văn bản vừa sinh vào mô hình MulCo để đưa ra dự đoán phân loại bệnh cuối cùng.

In [1]:
import os
import sys
import time
from pathlib import Path
import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms
from transformers import CLIPTokenizer, CLIPModel, AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score

# Tự động tìm thư mục gốc (chứa src)
current_dir = Path.cwd()
PROJECT_ROOT = current_dir
while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))
print(f"Project Root: {PROJECT_ROOT}")

from src.datasets.multimodal_raw_dataset import MultiModalRawDataset
from src.models.backbones.vision.convnext_cbam import ConvNeXt_CBAM
from src.models.fusion.mulco_fusion import MulCoFusionBlock
from src.models.multimodal.mulco_classifier import Conv1x1Classifier

Project Root: /media/data3/users/luongdth/MulCo-PlantNet


## 1. Cấu hình & Định nghĩa Model MulCo

In [2]:
class MulCoEndToEnd(nn.Module):
    def __init__(self, num_classes=28, proj_dim=512, spatial_size=(7, 7)):
        super().__init__()
        self.image_backbone = ConvNeXt_CBAM(num_classes=num_classes)
        self.text_backbone = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").text_model
        
        self.img_proj = nn.Conv2d(1024, proj_dim, kernel_size=1)
        self.txt_proj = nn.Linear(512, proj_dim)
        
        self.fusion_blocks = nn.ModuleList([
            MulCoFusionBlock(dim=proj_dim, num_heads=8) for _ in range(3)
        ])
        
        self.classifier = Conv1x1Classifier(in_channels=proj_dim, num_classes=num_classes, spatial_size=spatial_size)

    def forward(self, images, input_ids, attention_mask):
        img_feat = self.image_backbone.forward_features_spatial(images) 
        txt_out = self.text_backbone(input_ids=input_ids, attention_mask=attention_mask)
        txt_feat = txt_out.last_hidden_state
        
        img_feat = self.img_proj(img_feat)
        txt_feat = self.txt_proj(txt_feat)
        
        for block in self.fusion_blocks:
            img_feat, txt_feat = block(img_feat, txt_feat)
            
        return self.classifier(img_feat)

## 2. Load Models (LLaVA & MulCo)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Khởi tạo LLaVA
print("Loading LLaVA-1.5-7B...")
llava_processor = AutoProcessor.from_pretrained("llava-hf/llava-1.5-7b-hf")
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)
llava_model = LlavaForConditionalGeneration.from_pretrained(
    "llava-hf/llava-1.5-7b-hf",
    quantization_config=quantization_config,
    device_map="auto"
)

# Khởi tạo MulCo
print("Loading MulCo...")
mulco_model = MulCoEndToEnd(num_classes=28).to(device)
ckpt_path = os.path.join(PROJECT_ROOT, "archive", "cross_attention_3blocks", "best_model.pth")
mulco_model.load_state_dict(torch.load(ckpt_path))
mulco_model.eval()

# Bộ mã hóa văn bản của CLIP và Phép biến đổi ảnh
clip_tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
mulco_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

Using device: cuda
Loading LLaVA-1.5-7B...


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

Loading MulCo...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

## 3. Khởi tạo Pipeline Dự Đoán End-to-End

In [12]:
def predict_end_to_end(image_path):
    raw_image = Image.open(image_path).convert("RGB")
    
    # --- Bước 1: Sinh Text từ LLaVA ---
    prompt = "USER: <image>\nDescribe the characteristics and possible diseases of this plant leaf.\nASSISTANT:"
    inputs = llava_processor(text=prompt, images=raw_image, return_tensors="pt").to(device, torch.float16)
    
    with torch.no_grad():
        output = llava_model.generate(**inputs, max_new_tokens=100)
    generated_text = llava_processor.decode(output[0], skip_special_tokens=True)
    
    # Cắt bỏ phần prompt để lấy đúng câu trả lời
    caption = generated_text.split("ASSISTANT:")[-1].strip()
    
    # --- Bước 2: Tiền xử lý cho MulCo ---
    img_tensor = mulco_transform(raw_image).unsqueeze(0).to(device)
    
    text_tokens = clip_tokenizer(
        [caption], 
        padding=True, 
        truncation=True, 
        max_length=77, 
        return_tensors="pt"
    )
    input_ids = text_tokens.input_ids.to(device)
    attn_mask = text_tokens.attention_mask.to(device)
    
    # --- Bước 3: Đưa vào MulCo dự đoán ---
    with torch.no_grad():
        logits = mulco_model(img_tensor, input_ids, attn_mask)
        pred = torch.argmax(logits, dim=1).item()
        
    torch.cuda.empty_cache()
    return pred, caption

## 4. Chạy Đánh Giá Thực Tế

In [ ]:
# Sử dụng Dataset để lấy danh sách ảnh Val (bỏ qua json vì mình sẽ sinh caption bằng LLaVA)
test_dataset = MultiModalRawDataset(
    image_root=os.path.join(PROJECT_ROOT, "data/AIDG/dataset_PlantDoc/images/val"),
    caption_root=os.path.join(PROJECT_ROOT, "data/AIDG/captions_LLaVA/val"),
    transform=None,  # Để None vì ảnh sẽ được convert trong hàm predict_end_to_end
    use_depth_suppressed=False,
    strict_caption_match=False
)
print(f"Total items to test: {len(test_dataset)}")

all_preds = []
all_labels = []

# Mẹo: Ban đầu bạn nên đặt `num_samples_to_test = 10` để chạy thử xem code chạy ổn không.
# Nếu chạy ok, đổi sang `len(test_dataset)` để chạy hết.
num_samples_to_test = len(test_dataset)

for idx in tqdm(range(num_samples_to_test), desc="End-to-End Inference"):
    item = test_dataset[idx]
    image_path = item["image_path"]
    true_label = item["label"]
    
    start_time = time.time()
    pred, generated_caption = predict_end_to_end(image_path)
    latency = time.time() - start_time
    
    print(f"\n--- Image: {Path(image_path).name} | Time: {latency:.2f}s ---")
    print(f"Generated Caption:\n{generated_caption}")
    print(f"-> Predicted class: {pred} | True class: {true_label}")
    
    all_preds.append(pred)
    all_labels.append(true_label)

acc = accuracy_score(all_labels, all_preds)
print(f"\n====================================")
print(f"End-to-End Accuracy (on {len(all_labels)} samples): {acc:.4f}")
print(f"====================================")

[MultiModalRawDataset] Total selected images: 635
[MultiModalRawDataset] Valid samples: 635
[MultiModalRawDataset] Skipped missing caption: 0
[MultiModalRawDataset] Skipped invalid caption: 0
[MultiModalRawDataset] Matched by external mapping: 0
[MultiModalRawDataset] Num classes: 28
[MultiModalRawDataset] class_to_idx: {'Apple_Scab_Leaf': 0, 'Apple_leaf': 1, 'Apple_rust_leaf': 2, 'Bell_pepper_leaf': 3, 'Bell_pepper_leaf_spot': 4, 'Blueberry_leaf': 5, 'Cherry_leaf': 6, 'Corn_Gray_leaf_spot': 7, 'Corn_leaf_blight': 8, 'Corn_rust_leaf': 9, 'Peach_leaf': 10, 'Potato_leaf_early_blight': 11, 'Potato_leaf_late_blight': 12, 'Raspberry_leaf': 13, 'Soyabean_leaf': 14, 'Squash_Powdery_mildew_leaf': 15, 'Strawberry_leaf': 16, 'Tomato_Early_blight_leaf': 17, 'Tomato_Septoria_leaf_spot': 18, 'Tomato_leaf': 19, 'Tomato_leaf_bacterial_spot': 20, 'Tomato_leaf_late_blight': 21, 'Tomato_leaf_mosaic_virus': 22, 'Tomato_leaf_yellow_virus': 23, 'Tomato_mold_leaf': 24, 'Tomato_two_spotted_spider_mites_leaf'

End-to-End Inference:   0%|          | 0/635 [00:00<?, ?it/s]


--- Image: Apple_Scab_Leaf_00008.jpg | Time: 5.98s ---
Generated Caption:
The plant leaf in the image is yellow and has a few spots on it. These spots could be a sign of a disease or a natural occurrence, such as a fungal infection or a nutrient deficiency. It is important to identify the specific cause of the spots to determine the appropriate treatment or care for the plant. If the spots are a result of a disease, it is crucial to address the issue promptly to prevent further damage and maintain the health
-> Predicted class: 2 | True class: 2

--- Image: Apple_Scab_Leaf_00009.jpg | Time: 4.85s ---
Generated Caption:
The plant leaf in the image is yellow and has brown spots on it. These spots could be a sign of a disease or infection affecting the plant. The presence of brown spots on the leaf might indicate that the plant is experiencing stress or is affected by a fungal or bacterial infection. It is essential to identify the cause of the spots and take appropriate measures to addr